# 08 · Metastatic-cohort transfer test (local)

Does training platinum XGBoost on the metastatic ADT cohort pick up subtype signal that the full-cohort model misses?
Local scores are nested-CV out-of-fold (`cv_oof`); test-block rows are never used in evaluation. Logic lives in
`multivariable_models/local_endpoint_transfer.py`.

| Arm | Trained on | Scored on |
|---|---|---|
| A | platinum, metastatic cohort | metastatic patients |
| B | platinum, full cohort | the same metastatic patients (paired A − B) |
| C | = B | full cohort |
| F | frozen federated no-DFCI XGBoost (notebook 04 scores) | full / metastatic |
| N | NEPC, each cohort (`RUN_NEPC_CEILING`) | its own cohort; local ceiling |

`metastatic_adt` is retrospective (uses the full observed ADT course): a signal test, not a deployable model.

In [ ]:
import sys
sys.path.insert(0, "../pipeline")
import compass_pipeline as cp
for _p in (str(cp.PROJECT_ROOT),):
    if _p not in sys.path:
        sys.path.insert(0, _p)
from pathlib import Path
import pandas as pd
from survival_common import federated_endpoint_transfer as fet
from COMPASS.multivariable_models import local_endpoint_transfer as let

COHORT = "metastatic_adt"          # or "metastatic_llm"
RUN_NEPC_CEILING = True            # also fit/score NEPC-trained XGBoost (arm N)
ENDPOINTS = ("nepc", "avpc", "platinum")
LANDMARKS = (0, 90, 180)
N_BOOTSTRAP = 200
SEED = 0

DATA_ROOT = Path(cp._PROFILE_OUTPUT_ROOT)
RHINO_ROOT = Path(DATA_ROOT) / "rhino_data_v2"
FED_SCORES = RHINO_ROOT / "federated_transfer_adt_no_DFCI" / "all_models" / "transfer_scores.csv"
OUT_DIR = DATA_ROOT / "local_endpoint_transfer"
cp.OUT_OF_FOLD_RISKS = True
cp.FORCE_RERUN = False             # completed tasks are skipped

In [ ]:
# Fit the XGBoost runs that are missing (completed ones are skipped).
RUNS = cp.make_endpoint_runs(["adt"], endpoints=("platinum", "nepc"),
                             cohorts=("all", COHORT), exclusions=("none",))
XGB_SPECS = [s for s in cp.MULTIVARIATE_TASK_SPECS if s[0] == "xgboost"]
for run in RUNS:
    if run["endpoint"] == "platinum" or RUN_NEPC_CEILING:
        cp._run_tasks(run, XGB_SPECS)
RUN = {(r["cohort"], r["endpoint"]): r for r in RUNS}

# Every arm needs held-out OOF scores; a stale fit without them needs a refit of that run.
cov = pd.concat([cp.summarize_patient_risks(r).assign(cohort=r["cohort"]) for r in RUNS
                 if r["endpoint"] == "platinum" or RUN_NEPC_CEILING], ignore_index=True)
coverage_cols = ["cohort", "endpoint", "landmark", "config", "status", "n_oof_patients", "n_oof_scored"]
xgb_cov = cov.loc[cov["model"].eq("xgboost")].reindex(columns=coverage_cols)
display(xgb_cov)
oof_counts = xgb_cov[["n_oof_patients", "n_oof_scored"]].fillna(0)
bad = xgb_cov.loc[(oof_counts["n_oof_patients"] <= 0)
                  | (oof_counts["n_oof_scored"] != oof_counts["n_oof_patients"])]
if not bad.empty:
    raise RuntimeError(
        "Missing or incomplete OOF scores; evaluation stopped. Refit the listed tasks "
        "with cp.OUT_OF_FOLD_RISKS=True and cp.FORCE_RERUN=True, then restore "
        "cp.FORCE_RERUN=False and rerun this cell.\n" + bad.to_string(index=False)
    )

In [ ]:
def paths(cohort, endpoint):
    r = RUN[(cohort, endpoint)]
    return {(lm, cfg): cp.patient_risk_path(r, "xgboost", lm, cfg) for lm in LANDMARKS for cfg in ("both", "baseline")}

local = fet.read_local_frame(DATA_ROOT / "longitudinal_prediction_data_adt.csv", ENDPOINTS)
lab_mapping = fet.load_lab_mapping(fet.DEFAULT_LAB_MAPPING)

ceiling = None
if RUN_NEPC_CEILING:
    ceiling = {(c, lm, cfg): p for c in ("all", COHORT) for (lm, cfg), p in paths(c, "nepc").items()}

res = let.evaluate_metastatic_transfer(
    local, lab_mapping, landmarks=LANDMARKS, endpoints=ENDPOINTS,
    met_paths=paths(COHORT, "platinum"), all_paths=paths("all", "platinum"),
    federated_scores_path=FED_SCORES if FED_SCORES.exists() else None,
    ceiling_paths=ceiling, n_bootstrap=N_BOOTSTRAP, seed=SEED)
OUT_DIR.mkdir(parents=True, exist_ok=True)
res.to_csv(OUT_DIR / f"metastatic_transfer_{COHORT}.csv", index=False)
print(f"wrote {OUT_DIR / f'metastatic_transfer_{COHORT}.csv'}")

## Results

C-index by endpoint × landmark. `comparison="local"` reports the primary paired A − B on shared metastatic patients and C on the full local cohort. Optional `comparison="federated"` rows compare against F on separately intersected patients; they never change the local results. `comparison="ceiling"` contains N on its own cohort.

The primary A − B table below uses only local comparisons; treat a CI spanning 0 as no evidence.

In [ ]:
show = res[~res["arm"].str.endswith("_baseline")]
display(show.pivot_table(index=["endpoint", "landmark", "comparison", "population", "n", "events"], columns="arm", values="c_index").round(3))
display(show[show["arm"].eq("A") & show["comparison"].eq("local")][["endpoint", "landmark", "n", "events", "n_in_met_cohort", "events_in_met_cohort",
                                    "c_index", "delta_c_vs_ref", "delta_c_vs_ref_lo", "delta_c_vs_ref_hi"]].round(3))